# Demo 07c — Boosting

**Class 07 · E4** (slides: `slides/07_ensembles.md`)

Bagging builds its members side by side and averages them. **Boosting builds them one after another, each one correcting the mistakes of the others.**

* **AdaBoost:** ten points, three rounds, every number printed (weights before and after, the stump, the error $\text{err}_b$, the member weight $\beta_b$); why $\beta_b = \frac12 \ln\frac{1 - \text{err}_b}{\text{err}_b}$ (the minimum of the exponential loss, checked numerically); AdaBoost with stumps written by hand against `AdaBoostClassifier` on the moons: boundary and example weights after round 1, 2, 5 and 50, test error against the rounds, the margin.
* **Gradient boosting:** the squared loss in two rounds by hand (eight points), the same algorithm on the noisy sine against `GradientBoostingRegressor`, the residual and the tree of each round; the pseudo-residual of the logistic loss computed with `jax.grad` (Class 02) and checked against $y - p$; shrinkage $\eta$ against the number of trees; early stopping with a validation set on the heart disease data.
* **Libraries:** `GradientBoosting`, `HistGradientBoosting`, XGBoost and LightGBM on California housing: fit time and test error.
* **A failure case:** label noise.

Symbols (deck): $B$ rounds, $\hat f_b$ the member of round $b$, $w_i^{(b)}$ weight of example $i$, $\text{err}_b$ weighted error, $\beta_b$ member weight, $F_b$ the additive model after round $b$, $r_i^{(b)}$ pseudo-residual, $\eta$ shrinkage (the learning rate of Class 02). AdaBoost uses the labels $-1/+1$ inside; outside, class 1 is $+1$. Every cell that prints a number of a slide starts with `slide: <frame title>`.

In [ ]:
import time
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed
from scipy.optimize import minimize_scalar
from sklearn.datasets import make_moons
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

from faa import boundaries, data, metrics, plotlib, synthetic

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_fmt_str_lengths(70)


plotlib.setup(module="07")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0
N_JOBS = 8


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line: the frame title of the deck."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    """Print a polars table with rounded floats."""
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))


moons_X, moons_y = make_moons(300, noise=0.3, random_state=SEED)  # the running example of the deck
X_tr, X_te, y_tr, y_te = data.split(moons_X, moons_y, SEED)  # 210 train, 90 test
LIM = boundaries.plane_limits(moons_X)

## E4 — AdaBoost on ten points

Ten points of the moons (noise 0.2, the seed was chosen so that three stumps classify all ten). All weights start at $1/10$. Each round fits a **stump** (one split, depth 1) with the current weights, computes the **weighted error** $\text{err}_b$ and the member weight $\beta_b = \frac12\ln\frac{1 - \text{err}_b}{\text{err}_b}$, and multiplies the weight of each point by $e^{-\beta_b y_i \hat f_b(x_i)}$ (then divides by the sum $Z$): points the stump got right shrink, points it got wrong grow.

* **Round 1:** the stump $x_1 \le 0.053$ is wrong on points 2 and 4: $\text{err}_1 = 0.200$, $\beta_1 = 0.693$, $Z = 0.800$. The two wrong points go from 0.1 to 0.25, the other eight to 0.0625: **the wrong points now hold exactly half of the weight**, so the next stump starts at the level of a coin.
* **Round 2:** $x_1 \le 1.058$ is wrong on points 3 and 8 (weight 0.0625 each): $\text{err}_2 = 0.125$, $\beta_2 = 0.973$.
* **Round 3:** $x_1 \le 0.306$ is wrong on six points of small weight: $\text{err}_3 = 0.214$, $\beta_3 = 0.650$.
* The final vote $\mathrm{sign}(\sum_b \beta_b \hat f_b)$ is right on **all ten points**, although no stump is (the best of them errs on 2 of 10).

**Where $\beta$ comes from.** The weighted exponential loss of round 1 as a function of $\beta$ is $L(\beta) = \sum_i w_i e^{-\beta y_i \hat f_b(x_i)}$. A numerical minimisation gives $\beta^* = 0.6931$, the formula gives 0.6931, and $L(\beta^*) = 2\sqrt{err(1 - err)} = 0.800$: the value of $Z$ above (the weights are normalised by exactly this minimum).

In [ ]:
slide("AdaBoost on Ten Points: Round by Round")
X10, y10 = make_moons(10, noise=0.2, random_state=2)  # a toy set chosen so that three stumps classify all ten points
t10 = 2 * y10 - 1  # AdaBoost works with labels -1 / +1 (class 1 of the deck is +1)
w = np.full(10, 0.1)
F10 = np.zeros(10)
rounds10 = []
for s in range(1, 4):
    stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X10, t10, sample_weight=w)
    h = stump.predict(X10)
    err = float(w[h != t10].sum())
    alpha = 0.5 * np.log((1 - err) / err)
    w_new = w * np.exp(-alpha * t10 * h)
    Z = w_new.sum()
    rounds10.append(
        (
            s,
            stump.tree_.feature[0],
            float(stump.tree_.threshold[0]),
            err,
            float(alpha),
            w.copy(),
            (w_new / Z).copy(),
            h.copy(),
        )
    )
    print(
        f"round {s}: stump on x{stump.tree_.feature[0] + 1} <= {stump.tree_.threshold[0]:.3f}; weighted error err = {err:.3f}; beta = 1/2 ln((1 - err)/err) = {alpha:.3f}; normaliser Z = {Z:.3f}"
    )
    w = w_new / Z
    F10 += alpha * h
show(pl.DataFrame({"i": range(1, 11), "x1": X10[:, 0], "x2": X10[:, 1], "y": t10}), 3)
for s, f, thr, err, alpha, w_before, w_after, h in rounds10:
    print(f"round {s}: weights before {np.round(w_before, 3).tolist()}")
    print(
        f"         wrong points {(np.flatnonzero(h != t10) + 1).tolist()}; weights after {np.round(w_after, 3).tolist()}"
    )
print(
    f"after round 1: the {int(np.sum(rounds10[0][7] != t10))} misclassified points hold {rounds10[0][6][rounds10[0][7] != t10].sum():.3f} of the weight (always 1/2: the next stump starts at the level of a coin)"
)
print(
    f"final vote sign(sum beta_b f_b) = {np.sign(F10).astype(int).tolist()}; true y = {t10.tolist()}; training accuracy {np.mean(np.sign(F10) == t10):.1f}"
)

In [ ]:
slide("Where beta Comes From: the Exponential Loss")
s, f, thr, err, alpha, w_before, w_after, h = rounds10[0]
loss = lambda a: float(np.sum(w_before * np.exp(-a * t10 * h)))  # noqa: E731
res = minimize_scalar(loss, bounds=(0, 5), method="bounded", options={"xatol": 1e-10})
print(
    f"round 1: exponential loss of the weighted vote as a function of beta: L(beta) = sum_i w_i exp(-beta y_i f_b(x_i)); err = {err:.3f}"
)
print(
    f"numerical minimiser beta* = {res.x:.4f}; formula 1/2 ln((1 - err)/err) = {alpha:.4f}; L(beta*) = {res.fun:.4f}; 2 sqrt(err (1 - err)) = {2 * np.sqrt(err * (1 - err)):.4f}"
)
show(
    pl.DataFrame(
        {
            "alpha": [0.0, 0.25, 0.5, float(alpha), 1.0, 1.5],
            "L(beta)": [loss(a) for a in (0.0, 0.25, 0.5, alpha, 1.0, 1.5)],
        }
    ),
    4,
)

## AdaBoost with stumps on the moons

The class `HandAdaBoost` is the algorithm of the ten points on the 210 training points of the moons. `AdaBoostClassifier` with 50 stumps gives the **same 90 test predictions** (agreement 1.000, test accuracy 0.889); its $\beta_b$ is twice ours (the SAMME convention drops the factor $\frac12$, the vote is the same).

* The weighted error of the stump grows from 0.167 (round 1) to 0.480 (round 50): later stumps are given the points the earlier ones got wrong, and they are almost coin flips on them. The member weight falls from 0.805 to 0.039.
* The weights **concentrate**: at round 50 the largest weight is 13.8 times the uniform weight $1/n$, and the weights are equivalent to 22.7 % of the points (the effective share).
* The boundary is a **sum of axis-parallel steps**: after round 1 one cut, after round 5 a staircase, after round 50 close to the curved boundary (test accuracy 0.744, 0.744, 0.844, 0.889 after rounds 1, 2, 5, 50).

Over ten seeds (mean test error): stumps reach 0.099 at about 58 rounds and stay flat afterwards (0.113 at 300 rounds) while their training error keeps falling to 0.018; trees of depth 3 reach 0.097 at 12 rounds, **fit the training set exactly** (0.000 from round 50 on) and their test error rises slightly to 0.114. AdaBoost with weak members is slow to overfit; with strong members it is not.

The **margin** of a training point is $y F_B(x) / \sum_b \beta_b \in [-1, 1]$: a confident right vote is near 1. After 5 rounds 10 % of the points have a margin $\le 0$ (smallest $-0.483$); after 50, 5.7 % ($-0.104$); after 300 none (smallest $+0.016$). Boosting keeps raising the **smallest** margin, even when the training error does not change any more, which is one explanation of why the test error does not rise.

In [ ]:
slide("AdaBoost with Stumps on the Moons: by Hand and in scikit-learn")


class HandAdaBoost:
    """Discrete AdaBoost (SAMME, 2 classes) with depth-1 trees. Labels are 0 / 1 outside, -1 / +1 inside.
    `weights_[s]` are the example weights AFTER round s (the ones that train round s + 1)."""

    def __init__(self, n_rounds: int = 50, max_depth: int = 1, seed: int = 0):
        self.n_rounds, self.max_depth, self.seed = n_rounds, max_depth, seed

    def fit(self, X, y):
        t = 2 * np.asarray(y) - 1
        w = np.full(len(t), 1 / len(t))
        self.stumps_, self.alphas_, self.errs_, self.weights_ = [], [], [], []
        for s in range(self.n_rounds):
            m = DecisionTreeClassifier(max_depth=self.max_depth, random_state=self.seed).fit(X, t, sample_weight=w)
            h = m.predict(X)
            err = float(np.clip(w[h != t].sum(), 1e-12, 1 - 1e-12))
            alpha = 0.5 * np.log((1 - err) / err)
            w = w * np.exp(-alpha * t * h)
            w /= w.sum()
            (
                self.stumps_.append(m),
                self.alphas_.append(float(alpha)),
                self.errs_.append(err),
                self.weights_.append(w.copy()),
            )
        return self

    def decision_function(self, X, upto: int | None = None):
        k = self.n_rounds if upto is None else upto
        return sum(a * m.predict(X) for a, m in zip(self.alphas_[:k], self.stumps_[:k]))

    def predict_proba(self, X, upto: int | None = None):
        p1 = 1 / (1 + np.exp(-2 * self.decision_function(X, upto)))  # the logistic link of the exponential loss
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)


class AtRound:
    """A view of a HandAdaBoost after `s` rounds, so that boundary_panel can draw it."""

    def __init__(self, model, s):
        self.model, self.s = model, s

    def predict_proba(self, X):
        return self.model.predict_proba(X, self.s)


ada = HandAdaBoost(50).fit(X_tr, y_tr)
sk = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=SEED).fit(
    X_tr, y_tr
)
agree = float(np.mean(ada.predict(X_te) == sk.predict(X_te)))
print(
    f"by hand, 50 stumps: train acc {np.mean(ada.predict(X_tr) == y_tr):.3f}, test acc {np.mean(ada.predict(X_te) == y_te):.3f}"
)
print(
    f"AdaBoostClassifier, 50 stumps: train acc {sk.score(X_tr, y_tr):.3f}, test acc {sk.score(X_te, y_te):.3f}; the two agree on {agree:.3f} of the test points"
)
print(
    f"first alphas by hand {np.round(ada.alphas_[:4], 3).tolist()}, scikit-learn {np.round(sk.estimator_weights_[:4], 3).tolist()} (scikit-learn: beta = ln((1 - err)/err), no 1/2 factor in SAMME)"
)
print(
    f"weighted error of the stump of round s: {', '.join(f'{e:.3f}' for e in ada.errs_[:5])}, ... {ada.errs_[-1]:.3f} at round 50 (every stump is a weak learner, only slightly better than 0.5)"
)
rows = []
for s in (1, 2, 5, 50):
    wts = ada.weights_[s - 1]
    rows.append(
        (
            s,
            float(ada.alphas_[s - 1]),
            float(wts.max() * len(wts)),
            float(1 / (len(wts) * np.sum(wts**2))),
            *metrics.scores(AtRound(ada, s), X_te, y_te),
        )
    )
show(
    pl.DataFrame(
        rows,
        schema=["round b", "beta_b", "max weight / (1/n)", "effective share of points", "test acc", "test log loss"],
        orient="row",
    )
)

In [ ]:
# plot e4_adaboost
fg, axs = boundaries.new_fig(1, 4, panel_h=1.25)
for ax, s in zip(axs, (1, 2, 5, 50)):
    sizes = np.clip(2 + 800 * ada.weights_[s - 1], 2, 45)
    boundaries.boundary_panel(
        ax, AtRound(ada, s), X_tr, y_tr, X_te, y_te, title=f"after round {s}", lim=LIM, sizes=sizes
    )
fg.subplots_adjust(wspace=0.06)
plotlib.save(fg, "e4_adaboost")

In [ ]:
slide("Test Error Against the Number of Rounds")


def ada_curve(seed, depth, T=300):
    X_, y_ = make_moons(300, noise=0.3, random_state=seed)
    a, b, c, d = data.split(X_, y_, seed)
    m = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=depth), n_estimators=T, random_state=seed).fit(
        a, c
    )
    tr = np.array([np.mean(p != c) for p in m.staged_predict(a)])
    te = np.array([np.mean(p != d) for p in m.staged_predict(b)])
    return tr, te


curves = {}
for depth in (1, 3):
    r = Parallel(n_jobs=N_JOBS)(delayed(ada_curve)(s, depth) for s in range(10))
    curves[depth] = (np.mean([x[0] for x in r], axis=0), np.mean([x[1] for x in r], axis=0))
rows = []
for depth, (tr, te) in curves.items():
    for T in (1, 5, 10, 50, 100, 300):
        rows.append((depth, T, float(tr[T - 1]), float(te[T - 1])))
show(pl.DataFrame(rows, schema=["base tree depth", "rounds T", "train error", "test error"], orient="row"))
print(
    f"stumps: best mean test error {curves[1][1].min():.3f} at T = {int(curves[1][1].argmin()) + 1}; depth 3: best {curves[3][1].min():.3f} at T = {int(curves[3][1].argmin()) + 1}; training error of depth 3 reaches {curves[3][0][-1]:.3f}"
)

In [ ]:
# plot e4_rounds
fg, ax = plotlib.fig("full", 1.55)
T_axis = np.arange(1, 301)
for depth, col, name in ((1, BLUE, "stumps"), (3, ORANGE, "trees of depth 3")):
    tr, te = curves[depth]
    ax.plot(T_axis, te, color=col, label=f"{name}: test")
    ax.plot(T_axis, tr, color=col, ls="--", label=f"{name}: train")
ax.set(xscale="log", xlabel="boosting rounds $B$ (log scale)", ylabel="error rate (mean of 10 seeds)", ylim=(0, 0.3))
ax.legend(ncols=2, loc="upper right", fontsize=6.3, handlelength=1.6, columnspacing=0.8, labelspacing=0.25)
plotlib.save(fg, "e4_rounds")

In [ ]:
slide("The Margin of the Training Points")
t_tr = 2 * y_tr - 1
big = HandAdaBoost(300, max_depth=1).fit(X_tr, y_tr)
stump_out = np.array([m.predict(X_tr) for m in big.stumps_])  # 300 x n
cum = np.cumsum(np.array(big.alphas_)[:, None] * stump_out, axis=0)
norm = np.cumsum(big.alphas_)[:, None]
marg = {s: t_tr * cum[s - 1] / norm[s - 1] for s in (5, 50, 300)}
rows = [(s, float(np.mean(m <= 0)), float(np.min(m)), float(np.median(m))) for s, m in marg.items()]
show(
    pl.DataFrame(rows, schema=["rounds B", "share with margin <= 0", "smallest margin", "median margin"], orient="row")
)

In [ ]:
# plot e4_margin
fg, ax = plotlib.fig("col", 1.5)
for s, col in zip(marg, (SKY, BLUE, ORANGE)):
    m = np.sort(marg[s])
    ax.plot(m, np.arange(1, len(m) + 1) / len(m), color=col, label=f"$B$ = {s}")
ax.axvline(0, color="k", lw=0.6, ls=":")
ax.set(
    xlabel="normalised margin $y\\,F_B(x) / \\sum_b \\beta_b$", ylabel="share of training points below", xlim=(-1, 1)
)
ax.legend(loc="lower right", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e4_margin")

## E4 — Gradient boosting: fit the residuals

For the squared loss the negative gradient of $\frac12 (y - F)^2$ with respect to the prediction $F$ is the **residual** $r = y - F$. Gradient boosting starts from the mean $F_0 = \bar y$ and in round $b$ fits a small tree $\hat f_b$ to the residuals and adds $\eta h_s$.

**Two rounds by hand** (eight points of the noisy sine, stumps, $\eta = 0.5$): $F_0 = 0.149$; the first stump isolates the last point (split at $x \le 2.178$), its leaf value $-0.911$ is the mean residual of that point; the second splits at $x \le -2.068$. The training MSE goes 0.461, 0.372, 0.337: each round removes a part of what is left.

**On the noisy sine** (100 points, trees of depth 2, $\eta = 0.1$, 100 rounds) the class `HandGB` and `GradientBoostingRegressor` give the same predictions up to $10^{-15}$ and the same test RMSE (0.1719 inside the training range): scikit-learn runs this algorithm for the squared loss. The test RMSE against the noiseless sine falls 0.708 (the mean), 0.671, 0.645, 0.563, 0.448, 0.190, 0.172 after 0, 1, 2, 5, 10, 30, 100 rounds. In the figure the top row is $F_b$, the bottom row the residuals that the next tree is fitted to: at round 50 they look like noise (sd 0.170, the data noise is 0.3 in the training set because of the finite sample).

**Other losses, same loop.** The tree of round $b$ is fitted to the negative gradient of the loss. For the logistic loss with log-odds $F$ it is $y - p$ with $p = \sigma(F)$; `jax.grad` gives it without algebra (largest difference with the closed form $3 \cdot 10^{-7}$, float32). For the absolute loss the target is $\mathrm{sign}(y - F)$, which is why that loss is robust to outliers.

In [ ]:
slide("Gradient Boosting for Squared Loss: Two Rounds by Hand")


class HandGB:
    """Gradient boosting for squared loss: F_0 = mean(y); round s fits a tree to the residuals y - F_{s-1} and adds eta times it."""

    def __init__(self, n_rounds: int = 100, eta: float = 0.1, max_depth: int = 3, seed: int = 0):
        self.n_rounds, self.eta, self.max_depth, self.seed = n_rounds, eta, max_depth, seed

    def fit(self, X, y):
        self.f0_ = float(np.mean(y))
        F = np.full(len(y), self.f0_)
        self.trees_, self.residuals_ = [], []
        for s in range(self.n_rounds):
            r = y - F  # the pseudo-residual of the squared loss: minus the gradient of (y - F)^2 / 2
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth, criterion="friedman_mse", random_state=self.seed
            ).fit(X, r)
            F = F + self.eta * tree.predict(X)
            self.trees_.append(tree), self.residuals_.append(r)
        return self

    def predict(self, X, upto: int | None = None):
        k = self.n_rounds if upto is None else upto
        return self.f0_ + self.eta * sum(t.predict(X) for t in self.trees_[:k]) if k else np.full(len(X), self.f0_)


rng = np.random.default_rng(SEED)
x8 = np.sort(rng.uniform(-3, 3, 8))
y8 = np.sin(1.5 * x8) + rng.normal(0, 0.3, 8)
ETA8 = 0.5
g8 = HandGB(2, eta=ETA8, max_depth=1).fit(x8[:, None], y8)
F0 = np.full(8, g8.f0_)
h1, h2 = g8.trees_[0].predict(x8[:, None]), g8.trees_[1].predict(x8[:, None])
F1 = F0 + ETA8 * h1
F2 = F1 + ETA8 * h2
show(
    pl.DataFrame(
        {
            "x": x8,
            "y": y8,
            "F0": F0,
            "r1 = y - F0": g8.residuals_[0],
            "h1": h1,
            "F1": F1,
            "r2 = y - F1": g8.residuals_[1],
            "h2": h2,
            "F2": F2,
        }
    ),
    3,
)
print(
    f"eta = {ETA8}, stumps; F0 = mean(y) = {g8.f0_:.3f}; split of round 1 at x <= {g8.trees_[0].tree_.threshold[0]:.3f}, of round 2 at x <= {g8.trees_[1].tree_.threshold[0]:.3f}"
)
print(
    f"training MSE: F0 {np.mean((y8 - F0) ** 2):.3f}, F1 {np.mean((y8 - F1) ** 2):.3f}, F2 {np.mean((y8 - F2) ** 2):.3f}"
)

In [ ]:
slide("Gradient Boosting on the Noisy Sine: by Hand and in scikit-learn")
Xs_tr, ys_tr, Xs_te, ys_te, f_sine = synthetic.reg_data("sine", SEED, n_train=100)
inside = np.abs(Xs_te[:, 0]) <= 3
hand = HandGB(100, eta=0.1, max_depth=2).fit(Xs_tr, ys_tr)
skg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=SEED).fit(Xs_tr, ys_tr)
diff = float(np.max(np.abs(hand.predict(Xs_te) - skg.predict(Xs_te))))
print(
    f"100 trees of depth 2, eta = 0.1: by hand test RMSE inside {metrics.rmse(hand.predict(Xs_te[inside]), f_sine(Xs_te[inside, 0])):.4f}, scikit-learn {metrics.rmse(skg.predict(Xs_te[inside]), f_sine(Xs_te[inside, 0])):.4f}"
)
print(
    f"largest difference between the two predictions on {len(ys_te)} test points: {diff:.1e} (same algorithm, same trees)"
)
rows = [
    (
        s,
        float(np.mean((ys_tr - hand.predict(Xs_tr, s)) ** 2)),
        metrics.rmse(hand.predict(Xs_te[inside], s), f_sine(Xs_te[inside, 0])),
    )
    for s in (0, 1, 2, 5, 10, 30, 100)
]
show(pl.DataFrame(rows, schema=["rounds s", "train MSE", "test RMSE (inside)"], orient="row"), 4)

In [ ]:
# plot e4_residuals: F_s above, the residual and the tree fitted to it below
hand_r = HandGB(50, eta=0.3, max_depth=2).fit(Xs_tr, ys_tr)
xs = np.linspace(-3, 3, 300)[:, None]
fg, axs = plt.subplots(2, 4, figsize=(plotlib.FULL, 2.2), sharex=True)
order = np.argsort(Xs_tr[:, 0])
for j, s in enumerate((1, 2, 5, 50)):
    ax = axs[0, j]
    ax.scatter(Xs_tr[:, 0], ys_tr, s=3.5, color=ORANGE, edgecolor="white", linewidth=0.2, zorder=3)
    ax.plot(xs[:, 0], f_sine(xs[:, 0]), color=boundaries.LINE, lw=0.6, ls=":")
    ax.plot(xs[:, 0], hand_r.predict(xs, s), color=BLUE, lw=1.0)
    ax.set_xlabel("model $F$ and data", fontsize=6.3, labelpad=1.5)
    ax.set_title(f"after round {s}", fontsize=6.8, pad=2)
    ax = axs[1, j]
    r = hand_r.residuals_[s - 1]
    ax.scatter(Xs_tr[:, 0], r, s=3.5, color=GREY, edgecolor="white", linewidth=0.2, zorder=3)
    ax.plot(xs[:, 0], hand_r.trees_[s - 1].predict(xs), color=RED, lw=1.0)
    ax.axhline(0, color=boundaries.LINE, lw=0.4)
    ax.set_xlabel("residual and tree", fontsize=6.3, labelpad=1.5)
    for a in axs[:, j]:
        a.set(xticks=[], yticks=[])
        a.grid(False)
    axs[0, j].set_ylim(-1.6, 1.6)
    axs[1, j].set_ylim(-1.6, 1.6)
fg.subplots_adjust(wspace=0.06, hspace=0.38)
print(
    "residual sd per round (the part of y that the model has not explained):",
    ", ".join(f"s={s}: {hand_r.residuals_[s - 1].std():.3f}" for s in (1, 2, 5, 50)),
)
plotlib.save(fg, "e4_residuals")

In [ ]:
slide("Pseudo-Residuals of the Logistic Loss with jax.grad")


def logistic_loss(y, F):
    """-log-likelihood of a label y in {0, 1} for the log-odds F: softplus(F) - y F."""
    return jax.nn.softplus(F) - y * F


pseudo = jax.vmap(jax.grad(logistic_loss, argnums=1, allow_int=False), in_axes=(0, 0))
y5 = jnp.array([1.0, 1.0, 0.0, 0.0, 1.0])
F5 = jnp.array([-2.0, 0.0, 0.0, 1.5, 3.0])  # current log-odds
auto = -np.asarray(pseudo(y5, F5))  # r = - dL/dF
closed = np.asarray(y5 - jax.nn.sigmoid(F5))  # y - p
show(
    pl.DataFrame(
        {
            "y": np.asarray(y5),
            "F": np.asarray(F5),
            "p = sigmoid(F)": np.asarray(jax.nn.sigmoid(F5)),
            "r (jax.grad)": auto,
            "y - p": closed,
        }
    ),
    4,
)
Fbig = jnp.linspace(-6, 6, 1001)
auto_big = -np.asarray(jax.vmap(jax.grad(logistic_loss, argnums=1))(jnp.ones(1001), Fbig))
print(
    f"largest difference between jax.grad and y - p over F in [-6, 6]: {np.max(np.abs(auto_big - np.asarray(1 - jax.nn.sigmoid(Fbig)))):.1e}"
)
print(
    "squared loss: r = y - F (the residual); absolute loss: r = sign(y - F); logistic loss: r = y - p; the tree of round s is fitted to r in every case"
)
print(
    f"abs loss for the same F as a regression target y = 1, F = 0.2: r = {np.sign(1.0 - 0.2):+.0f}; squared loss: r = {1.0 - 0.2:+.1f}"
)

## Shrinkage $\eta$ and the number of trees

Depth-3 trees on 100 points of the noisy sine, mean of ten noise draws, RMSE inside the training range:

* $\eta = 1$ overfits at once: the best RMSE is 0.241 at 3 trees and it goes to 0.301 (the noise level of a model that has memorised the sample) by 500.
* $\eta = 0.3, 0.1, 0.03, 0.01$ all reach about **0.17** (0.172, 0.169, 0.170, 0.170), at 9, 31, 103 and 315 trees: **a smaller $\eta$ needs proportionally more trees and finds the same or a slightly better minimum**; the curve around the minimum is flatter, so the stopping point is easier to hit.
* Rule of thumb of the slides: a small $\eta$ (0.1 or less), enough trees, stop with a validation set.

In [ ]:
slide("Shrinkage and the Number of Trees")


def lr_curves(seed, etas=(1.0, 0.3, 0.1, 0.03, 0.01), T=500):
    a, b, c, d, f = synthetic.reg_data("sine", seed, n_train=100)
    inside_ = np.abs(c[:, 0]) <= 3
    out = {}
    for eta in etas:
        m = GradientBoostingRegressor(n_estimators=T, learning_rate=eta, max_depth=3, random_state=seed).fit(a, b)
        out[eta] = np.array([metrics.rmse(p[inside_], f(c[inside_, 0])) for p in m.staged_predict(c)])
    return out


lc = Parallel(n_jobs=N_JOBS)(delayed(lr_curves)(s) for s in range(10))
etas = list(lc[0])
mean_curve = {eta: np.mean([r[eta] for r in lc], axis=0) for eta in etas}
rows = []
for eta in etas:
    cur = mean_curve[eta]
    rows.append(
        (eta, float(cur[9]), float(cur[49]), float(cur[99]), float(cur[499]), float(cur.min()), int(cur.argmin()) + 1)
    )
show(
    pl.DataFrame(
        rows, schema=["eta", "10 trees", "50 trees", "100 trees", "500 trees", "best", "at tree"], orient="row"
    )
)
print(
    "test RMSE against the noiseless sine, inside the training range, mean of 10 noise draws; depth 3, 100 training points, noise sd 0.3"
)
print("a small eta needs more trees and gets a lower minimum; eta = 1 overfits after a few trees")

In [ ]:
# plot e4_lr_grid
fg, ax = plotlib.fig("full", 1.5)
for eta, col in zip(etas, (RED, ORANGE, GREEN, BLUE, PURPLE)):
    ax.plot(np.arange(1, 501), mean_curve[eta], color=col, label=f"$\\eta$ = {eta:g}")
ax.set(xscale="log", xlabel="trees $B$ (log scale)", ylabel="test RMSE", ylim=(0.1, 0.45))
ax.legend(ncols=5, loc="upper center", bbox_to_anchor=(0.5, 1.2), fontsize=6.3, handlelength=1.2, columnspacing=0.8)
plotlib.save(fg, "e4_lr_grid")

In [ ]:
slide("Early Stopping on a Validation Set")
Xh, yh = data.load_heart()


def early_run(seed, T=400):
    a, b, c, d = train_test_split(Xh, yh, test_size=0.4, stratify=yh, random_state=seed)
    a, v, c, e = train_test_split(a, c, test_size=0.25, stratify=c, random_state=seed)  # 60 / 20 / 20
    m = GradientBoostingClassifier(n_estimators=T, learning_rate=0.1, max_depth=3, random_state=seed).fit(a, c)
    curve = lambda X_, y_: np.array([log_loss(y_, p, labels=[0, 1]) for p in m.staged_predict_proba(X_)])  # noqa: E731
    es = GradientBoostingClassifier(
        n_estimators=T, learning_rate=0.1, max_depth=3, n_iter_no_change=10, validation_fraction=0.25, random_state=seed
    ).fit(a, c)
    return curve(a, c), curve(v, e), curve(b, d), es.n_estimators_, float(es.score(b, d)), float(m.score(b, d))


er = Parallel(n_jobs=N_JOBS)(delayed(early_run)(s) for s in range(10))
tr_c, va_c, te_c = (np.mean([r[i] for r in er], axis=0) for i in range(3))
best_s = int(va_c.argmin()) + 1
print("heart disease (303 patients), gradient boosting eta = 0.1, depth 3, 60/20/20 split, mean of 10 splits")
show(
    pl.DataFrame(
        [(s, float(tr_c[s - 1]), float(va_c[s - 1]), float(te_c[s - 1])) for s in (10, 25, best_s, 100, 200, 400)],
        schema=["trees B", "train log loss", "validation log loss", "test log loss"],
        orient="row",
    )
)
print(
    f"the mean validation loss is lowest at B = {best_s}; the test loss at that B is {te_c[best_s - 1]:.3f}, at B = 400 it is {te_c[-1]:.3f}"
)
print(
    f"n_iter_no_change = 10 (scikit-learn, its own validation share of 25 %) stops at {np.mean([r[3] for r in er]):.1f} trees on average (min {min(r[3] for r in er)}, max {max(r[3] for r in er)}); test accuracy {np.mean([r[4] for r in er]):.3f} against {np.mean([r[5] for r in er]):.3f} for all 400 trees"
)

In [ ]:
# plot e4_early
fg, ax = plotlib.fig("full", 1.5)
ax.plot(np.arange(1, 401), tr_c, color=BLUE, label="train")
ax.plot(np.arange(1, 401), va_c, color=ORANGE, label="validation")
ax.plot(np.arange(1, 401), te_c, color=GREY, ls="--", label="test")
ax.axvline(best_s, color=RED, lw=0.8, ls=":")
ax.annotate(
    f"stop at $B$ = {best_s}",
    (best_s, va_c[best_s - 1]),
    xytext=(1.6, 0.85),
    fontsize=6.5,
    color=RED,
    arrowprops={"arrowstyle": "-", "color": RED, "lw": 0.5},
)
ax.set(xscale="log", xlabel="trees $B$ (log scale)", ylabel="log loss", ylim=(0, 1.0))
ax.legend(ncols=3, loc="upper right", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e4_early")

## Early stopping and the libraries

**Early stopping.** Heart disease, 303 patients, 60/20/20 split, $\eta = 0.1$, depth 3, mean of ten splits. The training log loss falls to 0.000 at 400 trees while the validation loss has a minimum near **19 trees** (0.509) and then grows to 1.377; the test loss follows it (0.453 at 19 trees, 1.057 at 400). Stopping when the validation loss does not improve for 10 rounds (`n_iter_no_change=10` in scikit-learn) stops after 32 trees on average (14 to 54) and gives a test accuracy of 0.811 against 0.798 for all 400 trees.

**Libraries.** California housing, 16 512 training rows, 200 trees of depth 3, $\eta = 0.1$, four threads. The test RMSE is the same for the four implementations (about 0.50): they fit the same model with the same hyper-parameters. The fit time is not: the exact greedy trees of `GradientBoostingRegressor` need seconds, the histogram methods (`HistGradientBoostingRegressor`, XGBoost with `tree_method="hist"`, LightGBM) need about 0.1 s, **two orders of magnitude** less, because they sort each feature once into at most 255 bins. XGBoost and LightGBM add a regularised objective and (LightGBM) leaf-wise growth; CatBoost, which is not installed, orders the rows for its target encoding of categorical features. The exact times depend on the machine; the table printed below has this run's.

In [ ]:
slide("Modern Libraries: One Real Set, Four Implementations")
from lightgbm import LGBMRegressor  # noqa: E402
from xgboost import XGBRegressor  # noqa: E402

Xc, yc = data.load_california()
Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=0.2, random_state=SEED)
THREADS = 4
IMPLS = {
    "GradientBoosting (scikit-learn)": lambda: GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED
    ),
    "HistGradientBoosting (scikit-learn)": lambda: HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=3, max_leaf_nodes=None, random_state=SEED
    ),
    "XGBoost": lambda: XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, tree_method="hist", n_jobs=THREADS, random_state=SEED
    ),
    "LightGBM": lambda: LGBMRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, num_leaves=8, n_jobs=THREADS, random_state=SEED, verbose=-1
    ),
}
rows = []
for name, make in IMPLS.items():
    fits, preds = [], []
    for rep in range(1 if name.startswith("GradientBoosting") else 3):
        with threadpool_limits(THREADS):
            m = make()
            t0 = time.perf_counter()
            m.fit(Xa, ya)
            fits.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            p = m.predict(Xb)
            preds.append(time.perf_counter() - t0)
    rows.append((name, float(np.median(fits)), float(np.median(preds)), metrics.rmse(p, yb)))
timing = pl.DataFrame(rows, schema=["implementation", "fit s", "predict s", "test RMSE"], orient="row")
show(timing)
ref = timing["fit s"][0]
print(
    f"california housing: {len(ya)} training rows, {Xa.shape[1]} features, 200 trees of depth 3, eta = 0.1, {THREADS} threads (median of 3 runs, one run for the slow GradientBoosting; times depend on the machine)"
)
print(
    "fit time relative to GradientBoosting: "
    + ", ".join(
        f"{n.split(' ')[0]} {ref / t:.0f}x faster" for n, t in zip(timing["implementation"], timing["fit s"]) if t < ref
    )
)
print("CatBoost is not installed: it is on the slides only (ordered target statistics for categorical features)")

## Failure case: label noise (no slide)

A share of the **training** labels is flipped, the test labels stay clean. Moons with noise 0.2, 5 seeds: with clean labels the forest and the boosters are all at 0.94-0.95 (one tree 0.913). At 20 % flipped labels: forest 0.872, AdaBoost with stumps 0.883, gradient boosting 0.852, AdaBoost with depth-3 trees 0.837, one tree 0.757. At 30 %: stumps 0.848, forest 0.800, AdaBoost depth 3 0.770, gradient boosting 0.765. A flipped label is always misclassified, so boosting raises its weight round after round and the deep members memorise it; the forest averages it out. The stumps cannot memorise a point, so they stay the most robust member. Early stopping and a small $\eta$ are the remedy for the boosters.

In [ ]:
slide("Failure Case: Boosting and Label Noise (no slide)")


def noise_run(flip, seed):
    X_, y_ = make_moons(400, noise=0.2, random_state=seed)
    a, b, c, d = data.split(X_, y_, seed)
    rng = np.random.default_rng(seed + 100)
    c = np.where(rng.random(len(c)) < flip, 1 - c, c)  # flip a share of the training labels, test labels stay clean
    ms = {
        "AdaBoost, stumps": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=seed
        ),
        "AdaBoost, depth 3": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=3), n_estimators=200, random_state=seed
        ),
        "gradient boosting": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=seed),
        "random forest": RandomForestClassifier(n_estimators=200, random_state=seed),
        "one tree": DecisionTreeClassifier(random_state=seed),
    }
    with threadpool_limits(1):
        return [(flip, n, float(m.fit(a, c).score(b, d))) for n, m in ms.items()]


FLIPS = (0.0, 0.05, 0.1, 0.2, 0.3)
nr = Parallel(n_jobs=N_JOBS)(delayed(noise_run)(f, s) for f in FLIPS for s in range(5))
nl = pl.DataFrame([r for run in nr for r in run], schema=["flip", "model", "acc"], orient="row")
nw = nl.group_by("flip", "model").agg(pl.col("acc").mean()).pivot(on="model", index="flip", values="acc").sort("flip")
print("share of training labels flipped; clean test labels; mean test accuracy of 5 seeds:")
show(nw)
print(
    f"at 30 % flipped labels: forest {nw['random forest'][-1]:.3f}, gradient boosting {nw['gradient boosting'][-1]:.3f}, AdaBoost with depth-3 trees {nw['AdaBoost, depth 3'][-1]:.3f}, AdaBoost with stumps {nw['AdaBoost, stumps'][-1]:.3f}"
)
print(
    "boosting puts more weight on the points it gets wrong, and flipped labels are always wrong: the deep boosters lose more than the forest; stumps cannot memorise the noise, so they are the most robust here"
)

## Summary

* AdaBoost: $\text{err}_b$, $\beta_b = \frac12\ln\frac{1 - \text{err}_b}{\text{err}_b}$ (the minimiser of the exponential loss), weights $\times e^{-\beta_b y_i \hat f_b(x_i)}$; after each round the wrong points hold half of the weight.
* Gradient boosting: $F_s = F_{s-1} + \eta h_s$ with $\hat f_b$ fitted to the negative gradient of the loss (the residual for squared loss, $y - p$ for the logistic loss, found by `jax.grad`).
* Small $\eta$, enough trees, early stopping on a validation set.
* Histogram implementations are about 100 times faster than the exact one with the same accuracy.
* Boosting is sensitive to label noise; bagging and forests are not.
* Next: combining models of different kinds, and choosing among all of them on real data (`04_stacking_and_choice.ipynb`).